# 03 - KPIs Principais | Case Olist

Calcula os KPIs de Crescimento e Receita, Logistica e SLA, e Satisfacao,
a partir das tabelas em `data/processado/`. Fecha com um KPI executivo
que combina as duas trilhas.

Depende de `data/processado/` ja estar preenchido (notebook 02) e de
`data/base/olist_order_reviews_dataset.csv` (a nota de avaliacao ainda
nao faz parte do modelo dimensional, entao carregamos ela direto aqui).

In [1]:
import pandas as pd

## 0. Carregando os dados

In [2]:
fato_pedidos = pd.read_csv("../data/processado/fato_pedidos.csv")
dim_clientes = pd.read_csv("../data/processado/dim_clientes.csv")
dim_produtos = pd.read_csv("../data/processado/dim_produtos.csv")
dim_vendedores = pd.read_csv("../data/processado/dim_vendedores.csv")
order_reviews = pd.read_csv("../data/base/olist_order_reviews_dataset.csv")

fato_pedidos["order_purchase_timestamp"] = pd.to_datetime(fato_pedidos["order_purchase_timestamp"])
fato_pedidos["order_delivered_customer_date"] = pd.to_datetime(fato_pedidos["order_delivered_customer_date"])
fato_pedidos["order_estimated_delivery_date"] = pd.to_datetime(fato_pedidos["order_estimated_delivery_date"])
fato_pedidos["ano_mes"] = fato_pedidos["order_purchase_timestamp"].dt.strftime("%Y-%m")

print("Tabelas carregadas!")

Tabelas carregadas!


In [3]:
# Meses com volume quase nulo (documentado no dicionario de dados) -
# excluir das series temporais para nao distorcer a analise
meses_validos = fato_pedidos[(fato_pedidos["ano_mes"] >= "2017-01") & (fato_pedidos["ano_mes"] <= "2018-08")]
print("Linhas totais:", len(fato_pedidos), "| Linhas no periodo valido:", len(meses_validos))

Linhas totais: 112650 | Linhas no periodo valido: 112279


## 1. KPIs de Crescimento e Receita

### Receita total

In [4]:
receita_total = fato_pedidos["price"].sum()
print(f"Receita total: R$ {receita_total:,.2f}")

Receita total: R$ 13,591,643.70


### Evolucao mensal e taxa de crescimento (MoM)

In [5]:
receita_mensal = meses_validos.groupby("ano_mes")["price"].sum()
crescimento_mom = receita_mensal.pct_change() * 100

print(receita_mensal.round(0))
print("\nCrescimento medio mes a mes:", round(crescimento_mom.mean(), 1), "%")

ano_mes
2017-01     120313.0
2017-02     247303.0
2017-03     374344.0
2017-04     359927.0
2017-05     506071.0
2017-06     433039.0
2017-07     498031.0
2017-08     573972.0
2017-09     624402.0
2017-10     664219.0
2017-11    1010271.0
2017-12     743914.0
2018-01     950030.0
2018-02     844179.0
2018-03     983213.0
2018-04     996648.0
2018-05     996518.0
2018-06     865124.0
2018-07     895507.0
2018-08     854686.0
Name: price, dtype: float64

Crescimento medio mes a mes: 14.2 %


### Ticket medio mensal (AOV)

In [6]:
pedidos_unicos_mes = meses_validos.drop_duplicates(subset="order_id")
ticket_medio_mensal = meses_validos.groupby("ano_mes")["price"].sum() / pedidos_unicos_mes.groupby("ano_mes")["order_id"].count()
print(ticket_medio_mensal.round(2))

ano_mes
2017-01    152.49
2017-02    142.70
2017-03    141.74
2017-04    150.53
2017-05    138.27
2017-06    134.61
2017-07    125.48
2017-08    133.70
2017-09    147.16
2017-10    145.41
2017-11    135.59
2017-12    132.27
2018-01    131.58
2018-02    126.11
2018-03    136.79
2018-04    143.73
2018-05    145.41
2018-06    140.44
2018-07    142.76
2018-08    132.47
dtype: float64


### Concentracao de receita por categoria

In [7]:
fato_com_categoria = fato_pedidos.merge(dim_produtos[["product_id", "product_category_name_english"]], on="product_id", how="left")
receita_categoria = fato_com_categoria.groupby("product_category_name_english")["price"].sum().sort_values(ascending=False)
top5_categorias = receita_categoria.head(5)

print(top5_categorias.round(0))
print(f"\nTop 5 categorias = {top5_categorias.sum() / receita_total * 100:.1f}% da receita total")

product_category_name_english
health_beauty            1258681.0
watches_gifts            1205006.0
bed_bath_table           1036989.0
sports_leisure            988049.0
computers_accessories     911954.0
Name: price, dtype: float64

Top 5 categorias = 39.7% da receita total


### Concentracao de receita por regiao

In [8]:
fato_com_estado = fato_pedidos.merge(dim_clientes[["customer_id", "customer_state"]], on="customer_id", how="left")
receita_estado = fato_com_estado.groupby("customer_state")["price"].sum().sort_values(ascending=False)

print(receita_estado.head(5).round(0))
print(f"\nSP = {receita_estado.get('SP', 0) / receita_total * 100:.1f}% da receita total")

customer_state
SP    5202955.0
RJ    1824093.0
MG    1585308.0
RS     750304.0
PR     683084.0
Name: price, dtype: float64

SP = 38.3% da receita total


### Concentracao de receita por seller

In [9]:
receita_seller = fato_pedidos.groupby("seller_id")["price"].sum().sort_values(ascending=False)
top10_sellers = receita_seller.head(10)

print(f"Top 10 sellers = {top10_sellers.sum() / receita_total * 100:.1f}% da receita total, de {fato_pedidos['seller_id'].nunique()} sellers no total")

Top 10 sellers = 13.1% da receita total, de 3095 sellers no total


### Taxa de recompra

In [10]:
total_pedidos_cliente = dim_clientes["customer_id"].nunique()
total_clientes_unicos = dim_clientes["customer_unique_id"].nunique()
taxa_recompra = (1 - total_clientes_unicos / total_pedidos_cliente) * 100

print(f"Taxa de recompra: {taxa_recompra:.2f}%")

Taxa de recompra: 3.36%


## 2. KPIs de Logistica e SLA

Calculados por pedido (nao por item), entao removemos linhas repetidas de pedidos com mais de um item antes de tirar as medias.

In [11]:
pedidos_entregues = fato_pedidos[fato_pedidos["order_status"] == "delivered"].drop_duplicates(subset="order_id").copy()
pedidos_entregues["lead_time_dias"] = (pedidos_entregues["order_delivered_customer_date"] - pedidos_entregues["order_purchase_timestamp"]).dt.days
pedidos_entregues["atraso_dias"] = (pedidos_entregues["order_delivered_customer_date"] - pedidos_entregues["order_estimated_delivery_date"]).dt.days

print("Pedidos entregues (unicos):", len(pedidos_entregues))

Pedidos entregues (unicos): 96478


### Lead time medio

In [12]:
print("Lead time medio:", round(pedidos_entregues["lead_time_dias"].mean(), 1), "dias")

Lead time medio: 12.1 dias


### Percentual de pedidos no prazo (SLA compliance)

In [13]:
pct_no_prazo = (~pedidos_entregues["atrasado"]).mean() * 100
print(f"Pedidos no prazo: {pct_no_prazo:.1f}%")

Pedidos no prazo: 91.9%


### Atraso medio, entre os pedidos atrasados

In [14]:
pedidos_atrasados = pedidos_entregues[pedidos_entregues["atrasado"]]
print("Atraso medio:", round(pedidos_atrasados["atraso_dias"].mean(), 1), "dias, em", len(pedidos_atrasados), "pedidos atrasados")

Atraso medio: 8.9 dias, em 7826 pedidos atrasados


### Lead time medio por regiao

In [15]:
pedidos_com_estado = pedidos_entregues.merge(dim_clientes[["customer_id", "customer_state"]], on="customer_id", how="left")
lead_time_por_estado = pedidos_com_estado.groupby("customer_state")["lead_time_dias"].mean().sort_values(ascending=False)

print(lead_time_por_estado.head(5).round(1))

customer_state
RR    29.0
AP    26.7
AM    26.0
AL    24.0
PA    23.3
Name: lead_time_dias, dtype: float64


### Frete medio e frete/preco

In [16]:
frete_medio = fato_pedidos["freight_value"].mean()
frete_sobre_preco = fato_pedidos["freight_value"].sum() / fato_pedidos["price"].sum() * 100

print(f"Frete medio: R$ {frete_medio:.2f}")
print(f"Frete / preco: {frete_sobre_preco:.1f}%")

Frete medio: R$ 19.99
Frete / preco: 16.6%


## 3. KPIs de Satisfacao

In [17]:
pedidos_com_nota = pedidos_entregues.merge(order_reviews[["order_id", "review_score"]], on="order_id", how="inner")
print("Pedidos com nota:", len(pedidos_com_nota))

Pedidos com nota: 96361


### Nota media geral

In [18]:
print("Nota media:", round(pedidos_com_nota["review_score"].mean(), 2))

Nota media: 4.16


### Nota media: no prazo vs. atrasado

In [19]:
nota_por_atraso = pedidos_com_nota.groupby("atrasado")["review_score"].mean()
print(nota_por_atraso.round(2))

atrasado
False    4.29
True     2.57
Name: review_score, dtype: float64


### Percentual de avaliacoes negativas (nota 1 ou 2)

In [20]:
pct_negativas = (pedidos_com_nota["review_score"] <= 2).mean() * 100
print(f"Avaliacoes negativas: {pct_negativas:.1f}%")

Avaliacoes negativas: 12.8%


## 4. KPI Executivo Combinado — Receita em Risco por Atraso Logistico

Ideia: pegar a receita dos pedidos atrasados e ponderar pela queda percentual na nota de avaliacao (proxy do risco de nao recompra). Transforma o achado estatistico em um numero financeiro.

In [21]:
nota_no_prazo = nota_por_atraso[False]
nota_atrasado = nota_por_atraso[True]
queda_percentual = 1 - (nota_atrasado / nota_no_prazo)

pedidos_atrasados_ids = pedidos_entregues[pedidos_entregues["atrasado"]]["order_id"]
receita_atrasados = fato_pedidos[fato_pedidos["order_id"].isin(pedidos_atrasados_ids)]["price"].sum()

receita_em_risco = receita_atrasados * queda_percentual

print(f"Receita em pedidos atrasados: R$ {receita_atrasados:,.2f}")
print(f"Queda percentual na nota: {queda_percentual*100:.1f}%")
print(f"Receita em risco: R$ {receita_em_risco:,.2f}")

Receita em pedidos atrasados: R$ 1,158,920.51


Queda percentual na nota: 40.2%
Receita em risco: R$ 466,199.44


## Notas

- Todos os KPIs de Crescimento usam `price` (receita de itens), sem incluir frete.
- KPIs de Logistica e Satisfacao sao calculados por pedido unico, nao por item,
  para nao inflar pedidos com mais de um item.
- `order_reviews` foi carregado direto de `data/base/` porque a nota de
  avaliacao ainda nao faz parte do modelo dimensional em `data/processado/`.